In [1]:
library(dplyr)
library(data.table)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘data.table’


The following objects are masked from ‘package:dplyr’:

    between, first, last




In [31]:
tf <- fread("../data/barcode_calling/addgene.morf.library.info.csv")
input.tfs <- fread("../data/barcode_calling/input.orf.ids.txt", header = FALSE)
tf <- tf[Name %in% input.tfs$V1]
tf[, refseq := tstrsplit(.SD[[1]], ",")[[1]], .SDcols = 3]
tf[, fullname := paste0(.SD[[2]], "_", .SD[[1]], "_", .SD[[ncol(tf)]])]


In [37]:
fasta_lines <- paste0(">", tf$fullname, "\n", tf$`Full sequence`)

In [ ]:
writeLines(fasta_lines, "../data/barcode_calling/custom.tf.fasta")

In [48]:
head(nchar(tf$`Full sequence`))

[1] 9817 9814 9730 9130 8908 9430

In [39]:
#now let's turn this into a GTF 

In [55]:
gtf <- data.table(
  seqname = tf$fullname,
  source = "custom",
  feature = c("exon"),
  start = 1,
  end = nchar(tf$`Full sequence`),
  score = ".",
  strand = "+",
  frame = ".",
  attribute = paste0('gene_id ', tf$fullname)
)


In [52]:
head(gtf)

seqname,source,feature,start,end,score,strand,frame,attribute
<chr>,<chr>,<chr>,<dbl>,<int>,<chr>,<chr>,<chr>,<chr>
IRX5_TFORF0016_NM_005853,custom,gene,1,9817,.,+,.,gene_id IRX5_TFORF0016_NM_005853
IRX5_TFORF0017_NM_001252197,custom,gene,1,9814,.,+,.,gene_id IRX5_TFORF0017_NM_001252197
MAZ_TFORF0052_NM_001276275,custom,gene,1,9730,.,+,.,gene_id MAZ_TFORF0052_NM_001276275
BARX1_TFORF0122_NM_021570,custom,gene,1,9130,.,+,.,gene_id BARX1_TFORF0122_NM_021570
THAP6_TFORF0152_NM_001317791,custom,gene,1,8908,.,+,.,gene_id THAP6_TFORF0152_NM_001317791
EWSR1_TFORF0159_NM_001163287,custom,gene,1,9430,.,+,.,gene_id EWSR1_TFORF0159_NM_001163287


In [56]:
fwrite(gtf, "../data/barcode_calling/custom.tf.gtf", sep = "\t", col.names = FALSE, quote = FALSE)

In [58]:
#For small genomes, the parameter --genomeSAindexNbases must to be scaled down, with a typical
#value of min(14, log2(GenomeLength)/2 - 1). For example, for 1 megaBase genome, this is equal
#to 9, for 100 kiloBase genome, this is equal to 7.

min(14, log2(799909)/2 - 1)

[1] 8.804738